# Mission 3 — 2-1 실험: W0/W1 반복 seed (0.5 고정)

학교 계정 Colab GPU에서 실행하는 **Drive 미사용** 버전입니다. 첫 실행 때 Colab 파일
패널의 업로드 버튼으로 다음 두 파일을 `/content`에 올리세요.

1. `mission3_train_json.zip` (Mac: `Data AI Creater Camp/data/`)
2. `m3_targeted_weight_results.tar.gz` (Mac: `Downloads/`)

seed 42는 기존 결과 압축파일에서 재사용하고, seed 123·2026의 W0/W1 네 조건을
순서대로 학습합니다. 임계값은 0.5로 고정합니다. 같은 seed의 W0/W1은 손실 가중치만
다릅니다. 체크포인트는 매 epoch `/content/m3_2_1_targeted_weight/`에 저장됩니다.

**중요:** `/content`는 Mac 로컬이나 Google Drive가 아닙니다. 브라우저 연결만 잠시
끊겼다가 **같은 VM**에 재연결되면 저장된 체크포인트에서 이어갈 수 있습니다.
하지만 Colab 런타임 VM이 종료·교체되면 체크포인트와 결과가 모두 사라집니다.
맥을 충전하고 켜 두어도 이 제한은 바뀌지 않습니다. 학습 완료 후 마지막 셀에서
결과 압축파일을 즉시 다운로드하세요. 브라우저의 자동 다운로드 허용 여부를 확인하세요.


In [ ]:
%pip install -q "transformers==5.16.1" "accelerate>=1,<2" "scikit-learn>=1.5,<2" "pandas>=2.2,<3" safetensors "joblib>=1.4,<2"


## 1. GPU 환경과 업로드 파일 확인


In [ ]:
from pathlib import Path
import json, zipfile, tarfile, hashlib, gc, inspect, platform, io
import importlib.metadata as im
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, precision_recall_fscore_support
from transformers import (AutoTokenizer, RobertaModel, RobertaPreTrainedModel,
                          TrainingArguments, Trainer, set_seed, RobertaConfig)
from transformers.modeling_outputs import SequenceClassifierOutput
from google.colab import files

TRAIN_ZIP = Path("/content/mission3_train_json.zip")
OLD_BACKUP = Path("/content/m3_targeted_weight_results.tar.gz")
OUT = Path("/content/m3_2_1_targeted_weight")
OUT.mkdir(parents=True, exist_ok=True)
MANIFEST = OUT / "split_manifest.csv"
TARGETS = ["고열","구토","두통","복통","어지러움","열상","오심","전신쇠약","호흡곤란"]
MODEL_NAME = "klue/roberta-base"
SEEDS_TO_RUN = [123, 2026]
assert torch.cuda.is_available(), "먼저 GPU 런타임을 선택하세요."
for path in [TRAIN_ZIP, OLD_BACKUP]:
    assert path.is_file(), f"Colab 파일 패널에서 업로드하세요: {path}"
ENV = {"python": platform.python_version(), **{n: im.version(n) for n in
       ["torch","transformers","accelerate","numpy","pandas","scikit-learn"]}}
print("GPU:", torch.cuda.get_device_name(0), "환경:", ENV)
print("임시 체크포인트 경로:", OUT)
with tarfile.open(OLD_BACKUP, "r:gz") as a:
    manifest_bytes = a.extractfile("split_manifest.csv").read()
    reference_config = json.load(a.extractfile("w0_current_weight/run_config.json"))
    seed42_w1_config = json.load(a.extractfile("w1_targeted_low_recall_x125/run_config.json"))
assert reference_config["initial_parameter_sha256"] == seed42_w1_config["initial_parameter_sha256"]
if MANIFEST.exists():
    assert MANIFEST.read_bytes() == manifest_bytes
else:
    MANIFEST.write_bytes(manifest_bytes)
REVISION = reference_config["model_revision"]
assert REVISION


## 2. 원본 분할·정답 복원


In [ ]:
rows = []
with zipfile.ZipFile(TRAIN_ZIP) as z:
    for name in sorted(z.namelist()):
        p = Path(name)
        if "__MACOSX" in p.parts or p.name.startswith("._") or p.suffix.lower() != ".json":
            continue
        d = json.loads(z.read(name))
        texts = [u.get("text", "") for u in d["utterances"]]
        assert all(isinstance(t, str) for t in texts)
        assert isinstance(d["symptom"], list)
        rows.append({"file_name": p.name, "text": " ".join(texts), "utterance_texts": texts,
                     **{t: int(t in d["symptom"]) for t in TARGETS}})
df = pd.DataFrame(rows)
split = pd.read_csv(MANIFEST)
assert len(df) == 29200 and not df.file_name.duplicated().any()
assert not split.file_name.duplicated().any()
assert set(split.file_name) == set(df.file_name)
assert set(split.split) == {"train","dev_valid"}
ordered = split.merge(df, on="file_name", how="left", validate="one_to_one", sort=False)
train = ordered[ordered.split == "train"].reset_index(drop=True)
valid = ordered[ordered.split == "dev_valid"].reset_index(drop=True)
assert (len(train), len(valid)) == (23408, 5792)
assert hashlib.sha256(MANIFEST.read_bytes()).hexdigest() == reference_config["manifest_sha256"]
print("내부 학습:", train.shape, "내부 검증:", valid.shape)
print("manifest SHA256:", hashlib.sha256(MANIFEST.read_bytes()).hexdigest())

# seed=42 결과의 정답·행 순서도 현재 검증셋과 일치해야 합니다.
for experiment in ["w0_current_weight", "w1_targeted_low_recall_x125"]:
    with tarfile.open(OLD_BACKUP, "r:gz") as a:
        b = a.extractfile(f"{experiment}/dev_predictions.npz").read()
    with np.load(io.BytesIO(b), allow_pickle=False) as z:
        assert list(z["targets"]) == TARGETS
        assert np.array_equal(z["file_names"], valid.file_name.to_numpy())
        assert np.array_equal(z["y_true"], valid[TARGETS].to_numpy())
print("기존 seed=42와 동일한 데이터·검증 순서 확인")


## 3. 동일 입력과 W0/W1 가중치


In [ ]:
import re, unicodedata
MODES = ["raw", "boundary", "boundary_normalized"]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION)
def normalize_utterance(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text)).strip()

def encode_call(utterances, mode):
    assert mode in MODES
    if mode == "raw":
        body = tokenizer(" ".join(utterances), add_special_tokens=False, truncation=False)["input_ids"]
    else:
        body = []
        for i, text in enumerate(utterances):
            if i:
                body.append(tokenizer.sep_token_id)
            if mode == "boundary_normalized":
                text = normalize_utterance(text)
            body.extend(tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"])
    full_length = len(body) + 2
    return [tokenizer.cls_token_id] + body[:510] + [tokenizer.sep_token_id], full_length

class Calls(Dataset):
    def __init__(self, frame, mode):
        encoded = [encode_call(u, mode) for u in frame.utterance_texts]
        self.ids = [x[0] for x in encoded]
        self.lengths = np.array([x[1] for x in encoded])
        self.labels = frame[TARGETS].to_numpy(dtype=np.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids": self.ids[i], "labels": self.labels[i]}

def collate(batch):
    x = tokenizer.pad([{"input_ids": b["input_ids"]} for b in batch], return_tensors="pt")
    x["labels"] = torch.tensor(np.stack([b["labels"] for b in batch]), dtype=torch.float32)
    return x

assert normalize_utterance("토할  것 같아요.\n토는 안 했어요.") == "토할 것 같아요. 토는 안 했어요."
assert normalize_utterance("오심 구토 아니요 없어요") == "오심 구토 아니요 없어요"
for utterances in train.utterance_texts.iloc[:30]:
    actual, _ = encode_call(utterances, "raw")
    expected = tokenizer(" ".join(utterances), truncation=True, max_length=512)["input_ids"]
    assert actual == expected, "P0가 이전 입력과 다릅니다."
for mode in MODES:
    ids_test, _ = encode_call(["", "토는 안 했어요.", "속이 울렁거려요."], mode)
    assert len(ids_test) <= 512 and ids_test[0] == tokenizer.cls_token_id

# 이 단계부터 사용하는 전처리는 boundary 하나뿐입니다.
train_ds, valid_ds = Calls(train, "boundary"), Calls(valid, "boundary")
y_train = train[TARGETS].to_numpy(dtype=np.float32)
positive = y_train.sum(axis=0)
negative = len(train) - positive
assert (positive > 0).all(), "양성 0개 클래스가 있습니다."
BASE_WEIGHTS = np.clip(np.sqrt(negative / positive), 1., 3.).astype(np.float32)
TARGETED_SYMPTOMS = ["두통", "오심", "전신쇠약"]
BOOST_FACTOR = 1.25
BOOST_CAP = 4.0
BOOSTED_WEIGHTS = BASE_WEIGHTS.copy()
for symptom in TARGETED_SYMPTOMS:
    i = TARGETS.index(symptom)
    BOOSTED_WEIGHTS[i] = min(BOOSTED_WEIGHTS[i] * BOOST_FACTOR, BOOST_CAP)

EXPERIMENTS = ["w0_current_weight", "w1_targeted_low_recall_x125"]
WEIGHTS_BY_EXPERIMENT = {"w0_current_weight": BASE_WEIGHTS,
                         "w1_targeted_low_recall_x125": BOOSTED_WEIGHTS}
weight_table = pd.DataFrame({"symptom": TARGETS, "positive": positive, "negative": negative,
    "w0_current_weight": BASE_WEIGHTS, "w1_targeted_weight": BOOSTED_WEIGHTS,
    "is_targeted": [t in TARGETED_SYMPTOMS for t in TARGETS]})
display(weight_table)
weight_table.to_csv(OUT / "training_pos_weights.csv", index=False)
(OUT / "weight_rule.json").write_text(json.dumps({"base_rule": "clip(sqrt(negative/positive),1,3)",
    "targeted_symptoms": TARGETED_SYMPTOMS, "boost_factor": BOOST_FACTOR, "boost_cap": BOOST_CAP,
    "source": "internal train only", "w0_weights": BASE_WEIGHTS.tolist(),
    "w1_weights": BOOSTED_WEIGHTS.tolist()}, ensure_ascii=False, indent=2))
np.testing.assert_allclose(BASE_WEIGHTS, reference_config["pos_weight"], rtol=0, atol=1e-7)
print("동일 boundary First-512 입력 준비:", len(train_ds), len(valid_ds))

# 고정 임계값 평가 함수: 학습·요약 셀보다 먼저 정의해야 합니다.
def label_report(y, probs):
    pred = probs >= .5
    p, r, f, support = precision_recall_fscore_support(
        y, pred, average=None, zero_division=0)
    return pd.DataFrame({"symptom": TARGETS, "precision": p, "recall": r,
                         "f1": f, "support": support,
                         "predicted_positive": pred.sum(axis=0)})


## 4. 모델·손실 함수 검증


In [ ]:
class SymptomModel(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        h = config.hidden_size
        self.head_type = getattr(config, "head_type", "cls")
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.dense = nn.Linear(h, h)
        self.out_proj = nn.Linear(h, config.num_labels)
        if self.head_type == "label_attention":
            self.attention = nn.Linear(h, config.num_labels, bias=False)
        # Keep loss weights as ordinary config values, not a tensor buffer subject
        # to from_pretrained's meta-device / missing-weight initialization.
        values = getattr(config, "loss_pos_weight", [1.] * config.num_labels)
        if len(values) != config.num_labels or any(not np.isfinite(v) or v <= 0 for v in values):
            raise ValueError("Invalid loss weights")
        config.loss_pos_weight = [float(v) for v in values]
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, labels=None):
        hidden = self.roberta(input_ids=input_ids, attention_mask=attention_mask,
                              return_dict=True).last_hidden_state
        mask = attention_mask.bool().clone()
        mask[:,0] = False
        last = attention_mask.sum(1).long() - 1
        mask[torch.arange(len(mask), device=mask.device), last] = False
        empty = ~mask.any(dim=1)
        mask[empty,0] = True
        if self.head_type == "cls":
            pooled = hidden[:,0]
        elif self.head_type == "mean":
            pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
        else:
            scores = self.attention(hidden).float().masked_fill(~mask.unsqueeze(-1), -1e9)
            weights = scores.softmax(dim=1).to(hidden.dtype)
            pooled = torch.einsum("blc,blh->bch", weights, hidden)
        x = self.dropout(torch.tanh(self.dense(self.dropout(pooled))))
        if self.head_type == "label_attention":
            logits = (x * self.out_proj.weight.unsqueeze(0)).sum(-1) + self.out_proj.bias
        else:
            logits = self.out_proj(x)
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Non-finite logits: stop before saving results")
        loss = None
        if labels is not None:
            if labels.shape != logits.shape or not ((labels == 0) | (labels == 1)).all():
                raise ValueError("Labels must be binary and match logits")
            # Construct on the actual device AFTER loading, on each forward pass.
            pos_weight = torch.tensor(self.config.loss_pos_weight, device=logits.device,
                                      dtype=torch.float32)
            loss = nn.functional.binary_cross_entropy_with_logits(
                logits.float(), labels.float(), pos_weight=pos_weight)
            if not torch.isfinite(loss) or loss.detach().item() > 100:
                raise FloatingPointError("Abnormal BCE loss; inspect initialization and labels")
        return SequenceClassifierOutput(loss=loss, logits=logits)

def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    return {"macro_f1": f1_score(p.label_ids, logits >= 0, average="macro", zero_division=0)}

# 다운로드·학습 없는 구조 검사
tiny = RobertaConfig(vocab_size=32, hidden_size=24, num_hidden_layers=1,
                     num_attention_heads=4, intermediate_size=48, num_labels=9)
for kind in ["cls","mean","label_attention"]:
    tiny.head_type = kind
    m = SymptomModel(tiny).eval()
    with torch.no_grad():
        v = m(torch.tensor([[0,5,6,2,1],[0,2,1,1,1]]),
              torch.tensor([[1,1,1,1,0],[1,1,0,0,0]]), torch.zeros(2,9))
    assert v.logits.shape == (2,9) and torch.isfinite(v.loss)
del m
print("head 구조 검사 OK")

# Real save/load regression test using a tiny local model (no download).
import tempfile
def verify_loss(model, batch, expected_weights):
    model.eval()
    with torch.no_grad():
        result = model(**batch)
        z, y = result.logits.float(), batch["labels"].float()
        w = torch.tensor(expected_weights, dtype=torch.float32, device=z.device)
        # Independent BCE identity, with no reference to model.config weights.
        manual = ((1-y)*nn.functional.softplus(z) + w*y*nn.functional.softplus(-z)).mean()
    torch.testing.assert_close(result.loss, manual, rtol=1e-5, atol=1e-6)
    assert model.config.loss_pos_weight == list(expected_weights)
    return result

for test_weights in [[1.]*9, [2.]*9]:
    tiny.head_type = "label_attention"
    tiny.loss_pos_weight = test_weights
    test_model = SymptomModel(tiny).eval()
    test_batch = {"input_ids": torch.tensor([[0,5,6,2],[0,7,8,2]]),
                  "attention_mask": torch.ones(2,4,dtype=torch.long),
                  "labels": torch.tensor([[1,0,1,0,1,0,1,0,1],[0,1,0,1,0,1,0,1,0]],dtype=torch.float32)}
    before = verify_loss(test_model, test_batch, test_weights)
    with tempfile.TemporaryDirectory() as test_dir:
        test_model.save_pretrained(test_dir)
        loaded = SymptomModel.from_pretrained(test_dir).eval()
        after = verify_loss(loaded, test_batch, test_weights)
        torch.testing.assert_close(before.logits, after.logits)
    del test_model, loaded
print("BCE independent calculation + save/load checks passed")

from transformers import TrainerCallback
class FiniteGradientCheck(TrainerCallback):
    def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
        for name, parameter in model.named_parameters():
            if parameter.grad is not None and not torch.isfinite(parameter.grad).all():
                raise FloatingPointError(f"Non-finite gradient: {name}")
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and metrics.get("eval_macro_f1") == 0:
            raise RuntimeError("Epoch evaluation F1 is zero. Stop and inspect; do not run L1 yet.")


## 5. seed 123·2026 순차 실행

같은 런타임에서 셀 실행이 중단됐다면 위부터 다시 실행해 최신 epoch 체크포인트에서 재개합니다. VM이 종료되면 `/content` 파일도 사라집니다.


In [ ]:
from transformers import TrainerCallback

class CheckpointReady(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        checkpoint = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        assert checkpoint.is_dir(), f"체크포인트 폴더 없음: {checkpoint}"
        (checkpoint / ".checkpoint_complete").write_text(str(state.global_step))

def completed_run(seed, experiment):
    path = OUT / f"seed_{seed}" / experiment
    required = [path / "metrics.json", path / "dev_predictions.npz", path / "run_config.json"]
    if not all(p.is_file() for p in required):
        return False
    with np.load(path / "dev_predictions.npz", allow_pickle=False) as z:
        assert list(z["targets"]) == TARGETS
        assert np.array_equal(z["file_names"], valid.file_name.to_numpy())
        assert np.array_equal(z["y_true"], valid[TARGETS].to_numpy())
        assert np.isfinite(z["probs"]).all()
        score = float(f1_score(z["y_true"], z["probs"] >= .5, average="macro", zero_division=0))
    metrics = json.loads((path / "metrics.json").read_text())
    assert abs(score - metrics["fixed_threshold_macro_f1"]) < 1e-9
    return True

def usable_checkpoints(folder):
    needed = {"trainer_state.json", "optimizer.pt", "scheduler.pt", "rng_state.pth",
              ".checkpoint_complete"}
    for p in sorted(folder.glob("checkpoint-*"),
                    key=lambda x: int(x.name.split("-")[-1]), reverse=True):
        if needed.issubset({x.name for x in p.iterdir()}) and any(
            (p / n).is_file() and (p / n).stat().st_size > 100_000_000
            for n in ["model.safetensors", "pytorch_model.bin"]):
            yield p

def finish_saved_prediction(seed, experiment, out):
    """Finish a run that crashed after final predictions were saved."""
    prediction_path = out / "dev_predictions.npz"
    if not prediction_path.is_file():
        return False
    with np.load(prediction_path, allow_pickle=False) as z:
        assert list(z["targets"]) == TARGETS
        assert np.array_equal(z["file_names"], valid.file_name.to_numpy())
        assert np.array_equal(z["y_true"], valid[TARGETS].to_numpy())
        y, probs = z["y_true"].copy(), z["probs"].copy()
    assert probs.shape == y.shape and np.isfinite(probs).all()
    label_report(y, probs).to_csv(out / "per_label.csv", index=False)
    checkpoints = list(usable_checkpoints(out / "checkpoints"))
    if checkpoints:
        state = json.loads((checkpoints[0] / "trainer_state.json").read_text())
        (out / "training_log.json").write_text(json.dumps(state.get("log_history", []), indent=2))
    result = {"fixed_threshold": .5,
              "fixed_threshold_macro_f1": float(f1_score(y, probs >= .5, average="macro", zero_division=0)),
              "seed": seed, "experiment": experiment, "evaluation": "internal validation",
              "selection": "final_epoch"}
    temp_metrics = out / "metrics.tmp.json"
    temp_metrics.write_text(json.dumps(result, ensure_ascii=False, indent=2))
    temp_metrics.replace(out / "metrics.json")
    print("저장된 최종 예측으로 결과 복구:", result)
    return True

def run_experiment(seed, experiment):
    assert seed in SEEDS_TO_RUN and experiment in EXPERIMENTS
    out = OUT / f"seed_{seed}" / experiment
    out.mkdir(parents=True, exist_ok=True)
    if completed_run(seed, experiment):
        print("완료 결과 재사용:", out)
        return
    if finish_saved_prediction(seed, experiment, out):
        return
    checkpoints = list(usable_checkpoints(out / "checkpoints"))
    resume = str(checkpoints[0]) if checkpoints else None
    print("시작:", seed, experiment, "재개 위치:", resume or "처음부터")
    set_seed(seed)
    config = RobertaConfig.from_pretrained(MODEL_NAME, revision=REVISION)
    config.num_labels = len(TARGETS)
    config.id2label = dict(enumerate(TARGETS))
    config.label2id = {t:i for i,t in enumerate(TARGETS)}
    config.head_type = "label_attention"
    config.loss_pos_weight = WEIGHTS_BY_EXPERIMENT[experiment].tolist()
    model = SymptomModel.from_pretrained(MODEL_NAME, config=config, revision=REVISION)
    checks = sorted(set(int(np.flatnonzero(train[TARGETS].to_numpy()[:, j])[0]) for j in range(9)))
    batch = collate([train_ds[i] for i in checks])
    assert (batch["labels"].sum(0) > 0).all()
    verify_loss(model, batch, config.loss_pos_weight)
    model.train()
    digest = hashlib.sha256()
    for name, parameter in model.named_parameters():
        digest.update(name.encode())
        digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    initial_hash = digest.hexdigest()
    paired_hash_path = OUT / f"seed_{seed}" / "initial_parameter_sha256.txt"
    if paired_hash_path.exists():
        assert paired_hash_path.read_text() == initial_hash, "같은 seed의 W0/W1 초기값 불일치"
    else:
        paired_hash_path.write_text(initial_hash)
    kwargs = dict(output_dir=str(out / "checkpoints"), num_train_epochs=4,
                  per_device_train_batch_size=8, per_device_eval_batch_size=16,
                  gradient_accumulation_steps=1, learning_rate=2e-5, warmup_steps=1000,
                  weight_decay=.01, logging_steps=50, logging_first_step=True,
                  fp16=False, bf16=False, tf32=False, max_grad_norm=1.0,
                  report_to="none", seed=seed, data_seed=seed,
                  save_strategy="epoch", save_total_limit=1,
                  load_best_model_at_end=False)
    key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"
    kwargs[key] = "epoch"
    provenance = {"implementation": "loss_config_fp32_v2_targeted_weight_multiseed",
                  "head": "label_attention", "model": MODEL_NAME,
                  "model_revision": REVISION, "threshold": .5,
                  "loss": "weighted_bce", "seed": seed, "experiment": experiment,
                  "pos_weight": config.loss_pos_weight,
                  "initial_parameter_sha256": initial_hash,
                  "selection": "final_epoch", "input": "first_512",
                  "preprocessing": "boundary", "normalization": "none",
                  "targets": TARGETS, "manifest_sha256": hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
                  "training": kwargs}
    config_path = out / "run_config.json"
    if config_path.exists():
        assert json.loads(config_path.read_text()) == provenance, "저장된 실행 설정과 다릅니다. 중단하고 확인하세요."
    else:
        config_path.write_text(json.dumps(provenance, ensure_ascii=False, indent=2))
    trainer = Trainer(model=model, args=TrainingArguments(**kwargs), train_dataset=train_ds,
                      eval_dataset=valid_ds, data_collator=collate, compute_metrics=compute_metrics,
                      callbacks=[FiniteGradientCheck(), CheckpointReady()])
    trainer.train(resume_from_checkpoint=resume)
    logits = trainer.predict(valid_ds).predictions
    if isinstance(logits, tuple): logits = logits[0]
    probs = torch.sigmoid(torch.as_tensor(logits).float()).numpy()
    y = valid[TARGETS].to_numpy(dtype=int)
    assert probs.shape == y.shape and np.isfinite(probs).all()
    npz_temp = out / "dev_predictions.tmp.npz"
    np.savez_compressed(npz_temp, probs=probs, y_true=y,
                        file_names=valid.file_name.to_numpy(dtype=str), targets=np.asarray(TARGETS))
    npz_temp.replace(out / "dev_predictions.npz")
    label_report(y, probs).to_csv(out / "per_label.csv", index=False)
    (out / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2))
    result = {"fixed_threshold": .5,
              "fixed_threshold_macro_f1": float(f1_score(y, probs >= .5, average="macro", zero_division=0)),
              "seed": seed, "experiment": experiment, "evaluation": "internal validation",
              "selection": "final_epoch"}
    temp_metrics = out / "metrics.tmp.json"
    temp_metrics.write_text(json.dumps(result, ensure_ascii=False, indent=2))
    temp_metrics.replace(out / "metrics.json")  # 완료 표시를 마지막에 기록
    print("완료:", result)
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

for seed in SEEDS_TO_RUN:
    for experiment in EXPERIMENTS:
        run_experiment(seed, experiment)
print("모든 새 seed 학습 완료. 다음 결과 비교 셀을 실행하세요.")


## 6. 세 seed 비교

결과는 seed 42·123·2026의 W1−W0 paired 차이를 비교합니다.


In [ ]:
def load_result(seed, experiment):
    if seed == 42:
        with tarfile.open(OLD_BACKUP, "r:gz") as a:
            metrics = json.load(a.extractfile(f"{experiment}/metrics.json"))
            raw = a.extractfile(f"{experiment}/dev_predictions.npz").read()
        with np.load(io.BytesIO(raw), allow_pickle=False) as z:
            data = {k: z[k].copy() for k in z.files}
    else:
        assert completed_run(seed, experiment), f"미완료: {seed} {experiment}"
        path = OUT / f"seed_{seed}" / experiment
        metrics = json.loads((path / "metrics.json").read_text())
        with np.load(path / "dev_predictions.npz", allow_pickle=False) as z:
            data = {k: z[k].copy() for k in z.files}
    assert np.array_equal(data["file_names"], valid.file_name.to_numpy())
    assert np.array_equal(data["y_true"], valid[TARGETS].to_numpy())
    score = float(f1_score(data["y_true"], data["probs"] >= .5, average="macro", zero_division=0))
    assert abs(score - metrics["fixed_threshold_macro_f1"]) < 1e-9
    return data, score

rows, label_rows = [], []
for seed in [42] + SEEDS_TO_RUN:
    scores = {}
    for experiment in EXPERIMENTS:
        data, scores[experiment] = load_result(seed, experiment)
        report = label_report(data["y_true"], data["probs"])
        report.insert(0, "experiment", experiment)
        report.insert(0, "seed", seed)
        label_rows.append(report)
    rows.append({"seed": seed, "w0_macro_f1": scores[EXPERIMENTS[0]],
                 "w1_macro_f1": scores[EXPERIMENTS[1]],
                 "w1_minus_w0": scores[EXPERIMENTS[1]] - scores[EXPERIMENTS[0]]})
summary_df = pd.DataFrame(rows)
label_df = pd.concat(label_rows, ignore_index=True)
summary_df.to_csv(OUT / "seed_comparison.csv", index=False)
label_df.to_csv(OUT / "seed_per_label.csv", index=False)
display(summary_df)
print("평균 paired 차이:", summary_df.w1_minus_w0.mean())
print("paired 차이 표준편차:", summary_df.w1_minus_w0.std(ddof=1))
print("W1 우세 seed 수:", int((summary_df.w1_minus_w0 > 0).sum()), "/", len(summary_df))
display(label_df[label_df.symptom.isin(TARGETED_SYMPTOMS)].sort_values(["symptom", "seed", "experiment"]))
print("결과 파일:", OUT / "seed_comparison.csv", OUT / "seed_per_label.csv")


## 7. 결과를 Mac으로 다운로드

학습 완료 직후 실행하세요. 아래 파일은 지표·예측·설정만 포함하고 대형 체크포인트는 제외합니다.


In [ ]:
import tarfile
archive_path = Path("/content/m3_2_1_results.tar")
with tarfile.open(archive_path, "w") as archive:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and "checkpoints" not in path.relative_to(OUT).parts:
            archive.add(path, arcname=str(path.relative_to(OUT)))
print("결과 묶음 파일:", archive_path)
print("크기(GB):", round(archive_path.stat().st_size / 1e9, 2))
digest = hashlib.sha256()
with archive_path.open("rb") as f:
    for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
        digest.update(chunk)
print("SHA256:", digest.hexdigest())
files.download(str(archive_path))


## 8. 선택: 체크포인트 전체 다운로드

모델·옵티마이저 상태까지 보관하려면 실행하세요. 여러 GB일 수 있어 Colab 브라우저 다운로드가 실패할 수 있습니다. 먼저 7번의 작은 결과 파일부터 확보하세요.


In [ ]:
DOWNLOAD_CHECKPOINTS = False
if DOWNLOAD_CHECKPOINTS:
    big_archive = Path("/content/m3_2_1_checkpoints.tar")
    with tarfile.open(big_archive, "w") as archive:
        for path in sorted(OUT.rglob("*")):
            if path.is_file() and "checkpoints" in path.relative_to(OUT).parts:
                archive.add(path, arcname=str(path.relative_to(OUT)))
    print("체크포인트 크기(GB):", round(big_archive.stat().st_size / 1e9, 2))
    files.download(str(big_archive))
else:
    print("대형 체크포인트 다운로드를 건너뜁니다.")
